# Model Preprocessing
Preprocess train/test splits from the DW for Logistic Regression (LR) and Random Forest (RF).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLITS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'splits'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'model_ready'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'Diabetes_binary'
NUMERIC_COLS = ['BMI', 'PhysHlth', 'MentHlth']

In [2]:
train_lr = pd.read_csv(SPLITS_DIR / 'train_lr.csv')
test_lr = pd.read_csv(SPLITS_DIR / 'test_lr.csv')
train_rf = pd.read_csv(SPLITS_DIR / 'train_rf.csv')
test_rf = pd.read_csv(SPLITS_DIR / 'test_rf.csv')

print('LR train/test:', train_lr.shape, test_lr.shape)
print('RF train/test:', train_rf.shape, test_rf.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Final\\Dm_Final\\Diabetes-DW-Prediction\\data\\processed\\splits\\train_lr.csv'

In [ ]:
print('--- Raw numeric stats (train) ---')
pre_lr = train_lr[NUMERIC_COLS].describe().T
pre_rf = train_rf[NUMERIC_COLS].describe().T
print('LR raw:')
print(pre_lr.to_string())
print('\nRF raw:')
print(pre_rf.to_string())

In [ ]:
def fit_winsor_bounds(df: pd.DataFrame, cols: list[str], low_q: float = 0.01, high_q: float = 0.99):
    bounds = {}
    for col in cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if series.empty:
            bounds[col] = (np.nan, np.nan)
            continue
        bounds[col] = (float(series.quantile(low_q)), float(series.quantile(high_q)))
    return bounds


def apply_winsor(df: pd.DataFrame, bounds: dict[str, tuple[float, float]]):
    out = df.copy()
    for col, (lo, hi) in bounds.items():
        if col in out.columns and not (np.isnan(lo) or np.isnan(hi)):
            out[col] = out[col].clip(lower=lo, upper=hi)
    return out

## Logistic Regression preprocessing
- Scale numeric columns: BMI, PhysHlth, MentHlth
- Keep other columns as-is
- Save scaled datasets + scaler artifact (joblib)

In [ ]:
X_train_lr = train_lr.drop(columns=[TARGET])
y_train_lr = train_lr[TARGET].astype('int64')
X_test_lr = test_lr.drop(columns=[TARGET])
y_test_lr = test_lr[TARGET].astype('int64')

scaler = StandardScaler()
X_train_lr_scaled = X_train_lr.copy()
X_test_lr_scaled = X_test_lr.copy()

X_train_lr_scaled[NUMERIC_COLS] = scaler.fit_transform(X_train_lr[NUMERIC_COLS])
X_test_lr_scaled[NUMERIC_COLS] = scaler.transform(X_test_lr[NUMERIC_COLS])

train_lr_scaled = X_train_lr_scaled.copy()
train_lr_scaled[TARGET] = y_train_lr.values
test_lr_scaled = X_test_lr_scaled.copy()
test_lr_scaled[TARGET] = y_test_lr.values

train_lr_scaled.to_csv(OUT_DIR / 'train_lr_scaled.csv', index=False)
test_lr_scaled.to_csv(OUT_DIR / 'test_lr_scaled.csv', index=False)

dump(scaler, OUT_DIR / 'lr_scaler.joblib')

print('Saved LR scaled + scaler to:', OUT_DIR)

## Random Forest preprocessing
- Impute missing values (median) on numeric columns
- Winsorize numeric columns using train quantiles
- Save raw outputs + preprocessing artifacts (joblib)

In [ ]:
X_train_rf = train_rf.drop(columns=[TARGET])
y_train_rf = train_rf[TARGET].astype('int64')
X_test_rf = test_rf.drop(columns=[TARGET])
y_test_rf = test_rf[TARGET].astype('int64')

imputer = SimpleImputer(strategy='median')
X_train_rf_num = X_train_rf[NUMERIC_COLS]
X_test_rf_num = X_test_rf[NUMERIC_COLS]

X_train_rf_num = pd.DataFrame(
    imputer.fit_transform(X_train_rf_num),
    columns=NUMERIC_COLS,
    index=X_train_rf.index,
 )
X_test_rf_num = pd.DataFrame(
    imputer.transform(X_test_rf_num),
    columns=NUMERIC_COLS,
    index=X_test_rf.index,
 )

X_train_rf_imp = X_train_rf.copy()
X_test_rf_imp = X_test_rf.copy()
X_train_rf_imp[NUMERIC_COLS] = X_train_rf_num
X_test_rf_imp[NUMERIC_COLS] = X_test_rf_num

bounds = fit_winsor_bounds(X_train_rf_imp, NUMERIC_COLS, low_q=0.01, high_q=0.99)
X_train_rf_w = apply_winsor(X_train_rf_imp, bounds)
X_test_rf_w = apply_winsor(X_test_rf_imp, bounds)

train_rf_ready = X_train_rf_w.copy()
train_rf_ready[TARGET] = y_train_rf.values
test_rf_ready = X_test_rf_w.copy()
test_rf_ready[TARGET] = y_test_rf.values

train_rf_ready.to_csv(OUT_DIR / 'train_rf_ready.csv', index=False)
test_rf_ready.to_csv(OUT_DIR / 'test_rf_ready.csv', index=False)

dump({'imputer': imputer, 'winsor_bounds': bounds}, OUT_DIR / 'rf_preproc.joblib')

print('Saved RF ready + artifacts to:', OUT_DIR)

In [ ]:
print('--- Processed numeric stats (train) ---')
post_lr = train_lr_scaled[NUMERIC_COLS].describe().T
post_rf = train_rf_ready[NUMERIC_COLS].describe().T
print('LR scaled:')
print(post_lr.to_string())
print('\nRF ready (winsorized + imputed):')
print(post_rf.to_string())